# Stage 2 follow-up: Session Quality by Device Type and OS
## ISM 6562 — Final Project — MediStream Telehealth


## Setup

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .appName('MediStream-Stage2c-QualityByDeviceOS')
    .master('spark://spark-master:7077')
    .config('spark.executor.memory', '1g')
    .config('spark.driver.memory', '1g')
    .config('spark.sql.shuffle.partitions', '8')
    .getOrCreate())

HDFS_CURATED   = 'hdfs://namenode:9000/medistream/curated'
HDFS_ANALYTICS = 'hdfs://namenode:9000/medistream/analytics'
LOCAL_CURATED   = '/home/jovyan/data/curated'
LOCAL_ANALYTICS = '/home/jovyan/data/analytics'

try:
    spark.range(1).write.mode('overwrite').parquet(f'{HDFS_ANALYTICS}/_probe')
    CURATED, ANALYTICS = HDFS_CURATED, HDFS_ANALYTICS
    print('Using HDFS')
except Exception:
    CURATED, ANALYTICS = LOCAL_CURATED, LOCAL_ANALYTICS
    print('HDFS unavailable, using local mount')

print(f'  curated:   {CURATED}')
print(f'  analytics: {ANALYTICS}')

## Inspect schema and value distributions

Before aggregating, check the cardinality of `device_type` and `os` so we know what we're partitioning by.

In [ ]:
session = spark.read.parquet(f'{CURATED}/session_quality')
print(f'Loaded {session.count():,} session rows')
session.printSchema()

print('\ndevice_type distribution:')
session.groupBy('device_type').count().orderBy(F.desc('count')).show()

print('os distribution:')
session.groupBy('os').count().orderBy(F.desc('count')).show()

## Aggregate quality metrics by device_type × OS

Percentiles use `F.percentile_approx` (cheap and accurate enough for an analytics table). The `low_audio_pct` field flags the share of sessions where `audio_quality_score < 5` (mid-scale on the codec's 1–10 score) — a softer signal than the hard latency/packet-loss thresholds.

In [ ]:
LATENCY_THRESHOLD = 500
PACKET_LOSS_THRESHOLD = 5.0
LOW_AUDIO_THRESHOLD = 5

session_flagged = (session
    .withColumn('passes_latency', (F.col('latency_ms') <= LATENCY_THRESHOLD).cast('int'))
    .withColumn('passes_packet_loss', (F.col('packet_loss_pct') <= PACKET_LOSS_THRESHOLD).cast('int'))
    .withColumn('passes_quality',
        ((F.col('latency_ms') <= LATENCY_THRESHOLD) &
         (F.col('packet_loss_pct') <= PACKET_LOSS_THRESHOLD)).cast('int'))
    .withColumn('low_audio', (F.col('audio_quality_score') < LOW_AUDIO_THRESHOLD).cast('int'))
)

by_device_os = (session_flagged
    .groupBy('device_type', 'os')
    .agg(
        F.count('*').alias('session_count'),
        F.round(F.avg('latency_ms'), 2).alias('avg_latency_ms'),
        F.round(F.percentile_approx('latency_ms', 0.5), 2).alias('p50_latency_ms'),
        F.round(F.percentile_approx('latency_ms', 0.95), 2).alias('p95_latency_ms'),
        F.round(F.avg('packet_loss_pct'), 3).alias('avg_packet_loss_pct'),
        F.round(F.percentile_approx('packet_loss_pct', 0.95), 3).alias('p95_packet_loss_pct'),
        F.round(F.avg('audio_quality_score'), 2).alias('avg_audio_quality_score'),
        F.round(F.avg('low_audio') * 100, 2).alias('low_audio_pct'),
        F.round(F.avg('passes_quality') * 100, 2).alias('quality_pass_rate_pct'),
    )
    .orderBy('device_type', F.desc('session_count'))
)

out_path = f'{ANALYTICS}/quality_by_device_os'
(by_device_os.write
    .mode('overwrite')
    .partitionBy('device_type')
    .parquet(out_path))

print(f'Wrote {by_device_os.count():,} rows to {out_path}')

## Verify

In [ ]:
verify = spark.read.parquet(f'{ANALYTICS}/quality_by_device_os')
print(f'Read back {verify.count():,} rows, {len(verify.columns)} columns')
verify.printSchema()
print('\nWorst quality_pass_rate first (highest-impact platforms to fix):')
verify.orderBy('quality_pass_rate_pct').show(20, truncate=False)
print('\nBest quality_pass_rate first (reference baselines):')
verify.orderBy(F.desc('quality_pass_rate_pct')).show(10, truncate=False)